In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "rouge-score", "sentence-transformers"], check=True)

import glob, json, re, hashlib, shutil, time, statistics as st
from pathlib import Path
import torch
from tqdm.auto import tqdm
from datasets import load_from_disk, load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, LoraConfig

CK = os.path.dirname(glob.glob("/kaggle/input/**/experiment_checkpoint/experiment_metadata.json", recursive=True)[0])
P2 = os.path.dirname(glob.glob("/kaggle/input/**/weights.json", recursive=True)[0])
WORK = Path("/kaggle/working"); (WORK / "results").mkdir(exist_ok=True)
print("CK:", CK, "\nP2:", P2)

# ---- adapters (sha256 verified) ----
MAN = json.load(open(f"{CK}/adapters_manifest.json"))
BASE = "mistralai/Mistral-7B-Instruct-v0.2"
NAMES = ["json", "tone", "sql", "samsum"]
AD = WORK / "adapters"; AD.mkdir(exist_ok=True)
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""): h.update(c)
    return h.hexdigest()
local = {sha256(p): p for p in glob.glob("/kaggle/input/**/ai-fragment/*.safetensors", recursive=True)}
for n in ["json", "tone"]:
    (AD / n).mkdir(exist_ok=True); shutil.copy2(local[MAN[n]["sha256"]], AD / n / "adapter_model.safetensors")
    LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
               target_modules=MAN[n]["target_modules"], base_model_name_or_path=BASE).save_pretrained(AD / n)
for n in ["sql", "samsum"]:
    snapshot_download(MAN[n]["repo"], revision=MAN[n]["revision"], local_dir=str(AD / n),
                      allow_patterns=["adapter_config.json", "adapter_model.safetensors"])
for n in NAMES: assert sha256(AD / n / "adapter_model.safetensors") == MAN[n]["sha256"], n
print("4 adapters: sha256 MATCH")

# ---- base model + saare adapters ----
REV = json.load(open(f"{CK}/experiment_metadata.json"))["base_revision"]
tok = AutoTokenizer.from_pretrained(BASE, revision=REV)
if tok.pad_token is None: tok.pad_token = tok.eos_token
q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                       bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
base_model = AutoModelForCausalLM.from_pretrained(BASE, revision=REV, quantization_config=q,
                                                  device_map="auto", dtype=torch.float16)
model = PeftModel.from_pretrained(base_model, str(AD / "json"), adapter_name="json", is_trainable=False)
for n in ["tone", "sql", "samsum"]: model.load_adapter(str(AD / n), adapter_name=n, is_trainable=False)
model.eval()
print("adapters loaded:", list(model.peft_config), "| GPU GB:", round(torch.cuda.memory_allocated() / 1e9, 2))

CK: /kaggle/input/datasets/khushisoni007/fragment-phase1/experiment_checkpoint 
P2: /kaggle/input/datasets/khushisoni007/fragment-phase2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

4 adapters: sha256 MATCH


... (8 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

adapters loaded: ['json', 'tone', 'sql', 'samsum'] | GPU GB: 4.54


In [2]:
M4_W = json.load(open(f"{P2}/weights.json"))["M4_W"]
STD = {"base": BASE, "base_revision": REV, "r": 16, "alpha": 32,
       "targets": {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}}

SKILL = {
  "json":   {"type": "task",  "description": "Extract structured information from text into a JSON object that follows a given schema.",
             "prompt_format": "chat template; 'Extract the information... Schema: ... Text: ...'",
             "metric": "field-level F1", "base": 0.369, "solo": 0.662},
  "tone":   {"type": "style", "description": "Write a warm, empathetic customer-support reply to a customer's message.",
             "prompt_format": "chat template; 'Subject: ...\\n\\n<customer message>' (no instruction)",
             "metric": "reply loss (lower is better)", "base": 2.832, "solo": 1.663},
  "sql":    {"type": "task",  "description": "Convert a natural-language question about a database into an SQL query.",
             "prompt_format": "chat template; SuperNI task107 input (Definition + examples + question)",
             "metric": "ROUGE-L (EM also tracked)", "base": 0.500, "solo": 0.737},
  "samsum": {"type": "task",  "description": "Summarize a chat conversation between people in one or two sentences.",
             "prompt_format": "chat template; SuperNI task1572 Definition + examples, dialogue joined by ', '",
             "metric": "ROUGE-L", "base": 0.312, "solo": 0.418},
}

def compat_status(cfg):
    """FAIL = merge nahi ho sakta | ALIGNABLE = missing modules zero-fill se theek | OK = bilkul standard"""
    if cfg["base_model_name_or_path"] != STD["base"] or cfg["r"] != STD["r"] or cfg["lora_alpha"] != STD["alpha"]:
        return "FAIL"
    t = set(cfg["target_modules"])
    if not t <= STD["targets"]: return "FAIL"
    return "OK" if t == STD["targets"] else "ALIGNABLE"

REGISTRY = {"standard": {**STD, "targets": sorted(STD["targets"])}, "fragments": {}}
for n in NAMES:
    cfg = json.load(open(AD / n / "adapter_config.json"))
    REGISTRY["fragments"][n] = {
        "id": n, **SKILL[n],
        "source": MAN[n].get("repo", "in-house"), "revision": MAN[n].get("revision"),
        "sha256": MAN[n]["sha256"], "path": str(AD / n),
        "r": cfg["r"], "alpha": cfg["lora_alpha"], "target_modules": sorted(cfg["target_modules"]),
        "compat": compat_status(cfg),
        "merge_weight": round(M4_W[n], 3),            # M4 se seekha hua weight
    }
json.dump(REGISTRY, open(WORK / "registry.json", "w"), indent=2)

print(f"{'id':7} {'type':5} {'compat':9} {'weight':>6}  {'base':>6} -> {'solo':<6} metric")
for n, f in REGISTRY["fragments"].items():
    print(f"{n:7} {f['type']:5} {f['compat']:9} {f['merge_weight']:6.3f}  {f['base']:6.3f} -> {f['solo']:<6.3f} {f['metric']}")
assert all(f["compat"] != "FAIL" for f in REGISTRY["fragments"].values())
print("\nregistry saved -> /kaggle/working/registry.json")

id      type  compat    weight    base -> solo   metric
json    task  OK         0.538   0.369 -> 0.662  field-level F1
tone    style OK         0.377   2.832 -> 1.663  reply loss (lower is better)
sql     task  ALIGNABLE  0.941   0.500 -> 0.737  ROUGE-L (EM also tracked)
samsum  task  ALIGNABLE  0.674   0.312 -> 0.418  ROUGE-L

registry saved -> /kaggle/working/registry.json


In [3]:
import random, pickle, numpy as np
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

ENC = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda")
LABELS = ["json", "tone", "sql", "samsum", "none"]

JSON_TMPL = """Extract the information from the text as JSON that follows the schema.
Output JSON only, no explanation.

Instruction: Extract all the information that the schema asks for.

Schema:
{schema}

Text:
{text}"""
def ni_input(s):                      # SuperNI prompt ka asli sawaal (last "Input:")
    i = s.rfind("Input: "); j = s.find("\nOutput", i)
    return s[i + len("Input: "): j].strip()

# --- user ki taraf se aane wali query ka form ---
def q_json(ex):  return JSON_TMPL.format(schema=ex["schema"], text=ex["text"])
def q_sql(ex):   return ni_input(ex["input"])
def q_samsum(d): return d
def q_tone(ex):  return f"Subject: {(ex['subject'] or '(no subject)').strip()}\n\n{ex['body']}"
def q_none(e):   return (e["instruction"] + ("\n\n" + e["context"] if e["context"] else "")).strip()

# --- data ---
EVAL = {t: load_from_disk(f"{CK}/evaluation_datasets/{t}") for t in NAMES}
pj = load_dataset("paraloq/json_data_extraction")["train"].shuffle(seed=42)        # 0-70 test, 71-90 dev
sql_tr = load_dataset("Lots-of-LoRAs/task107_splash_question_to_sql")["train"]
sam_tr = load_dataset("knkarthick/samsum")["train"]
arp = load_dataset("arpieb/consumer-electronics-support"); arp = arp[list(arp.keys())[0]]
cands = []
for row in arp:
    t = row["turns"]
    if len(t) < 2 or t[-1]["role"] != "agent" or not any(x["role"] == "customer" for x in t[:-1]): continue
    cands.append({"id": row["record_id"], "subject": row["scenario"],
                  "body": "\n".join(f"{x['role'].capitalize()}: {x['content']}" for x in t[:-1])})
random.Random(42).shuffle(cands)                                                   # 0-149 test, 150-179 dev
GENERAL = {"open_qa", "general_qa", "brainstorming", "creative_writing", "classification"}
dolly = load_dataset("databricks/databricks-dolly-15k")["train"].filter(lambda e: e["category"] in GENERAL).shuffle(seed=42)

NTR, NTE = 40, 50
TRAIN = {"json":   [q_json(pj[i]) for i in range(91, 91 + NTR)],
         "sql":    [q_sql(x) for x in sql_tr.select(range(NTR))],
         "samsum": [q_samsum(x["dialogue"]) for x in sam_tr.select(range(NTR))],
         "tone":   [q_tone(c) for c in cands[180:180 + NTR]],
         "none":   [q_none(x) for x in dolly.select(range(NTR))]}
TEST = {"json":   [q_json(EVAL["json"][i]) for i in range(len(EVAL["json"])) if i != 56][:NTE],
        "sql":    [q_sql(x) for x in EVAL["sql"].select(range(NTE))],
        "samsum": [q_samsum(x["dialogue"]) for x in EVAL["samsum"].select(range(NTE))],
        "tone":   [q_tone(x) for x in EVAL["tone"].select(range(NTE))],
        "none":   [q_none(x) for x in dolly.select(range(NTR, NTR + NTE))]}
assert not set(TRAIN["json"]) & set(TEST["json"]) and not set(TRAIN["tone"]) & set(TEST["tone"]), "train/test overlap!"

# --- train ---
def emb(texts): return ENC.encode(texts, normalize_embeddings=True, batch_size=32, show_progress_bar=False)
Xtr = np.vstack([emb(TRAIN[l]) for l in LABELS]); ytr = sum([[l] * len(TRAIN[l]) for l in LABELS], [])
clf = LogisticRegression(max_iter=2000, C=10).fit(Xtr, ytr)

def route(query, thr=0.30, k=2):
    """Returns (chune gaye adapters, probabilities). Khaali list = sirf base model."""
    p = dict(zip(clf.classes_, clf.predict_proba(emb([query]))[0]))
    top = sorted(p.items(), key=lambda x: -x[1])
    if top[0][0] == "none": return [], p
    return [l for l, v in top if l != "none" and v >= thr][:k], p

# --- test ---
t0 = time.time(); y_true, y_pred = [], []
for l in LABELS:
    for qtxt in TEST[l]:
        chosen, _ = route(qtxt)
        y_true.append(l); y_pred.append(chosen[0] if chosen else "none")
lat_ms = (time.time() - t0) / len(y_true) * 1000

print(f"{'class':7} accuracy")
for l in LABELS:
    idx = [i for i, t in enumerate(y_true) if t == l]
    print(f"{l:7} {sum(y_pred[i] == l for i in idx) / len(idx):.2f}   (n={len(idx)})")
acc = sum(a == b for a, b in zip(y_true, y_pred)) / len(y_true)
print(f"\nOVERALL top-1 accuracy: {acc:.3f} | router latency: {lat_ms:.1f} ms/query")
print("\nconfusion matrix (rows = true, cols = predicted):", LABELS)
print(confusion_matrix(y_true, y_pred, labels=LABELS))

pickle.dump({"clf": clf, "encoder": "sentence-transformers/all-MiniLM-L6-v2", "labels": LABELS, "thr": 0.30, "k": 2},
            open(WORK / "router.pkl", "wb"))
json.dump({"y_true": y_true, "y_pred": y_pred, "accuracy": acc, "latency_ms": lat_ms},
          open(WORK / "results" / "router_eval.json", "w"))
print("router saved -> /kaggle/working/router.pkl")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (34 outputs left out)

databricks-dolly-15k.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Filter:   0%|          | 0/15011 [00:00<?, ? examples/s]

class   accuracy
json    1.00   (n=50)
tone    1.00   (n=50)
sql     0.92   (n=50)
samsum  0.94   (n=50)
none    0.86   (n=50)

OVERALL top-1 accuracy: 0.944 | router latency: 8.3 ms/query

confusion matrix (rows = true, cols = predicted): ['json', 'tone', 'sql', 'samsum', 'none']
[[50  0  0  0  0]
 [ 0 50  0  0  0]
 [ 0  0 46  0  4]
 [ 0  1  0 47  2]
 [ 0  0  6  1 43]]
router saved -> /kaggle/working/router.pkl


In [4]:
from safetensors.torch import load_file, save_file

REG = json.load(open(WORK / "registry.json"))["fragments"]
SCALE, R_MERGED, SUF_A, SUF_B = 32 / 16, 64, ".lora_A.weight", ".lora_B.weight"
W = {n: load_file(str(AD / n / "adapter_model.safetensors")) for n in NAMES}

def build_concat(weights, out_dir):
    """Exact weighted merge: sum_i w_i * 2.0 * B_i A_i  (rank 64 tak zero pad)."""
    prefixes = sorted({k[:-len(SUF_A)] for n in weights for k in W[n] if k.endswith(SUF_A)})
    out = {}
    for pre in prefixes:
        As = [W[n][pre + SUF_A].float() for n in weights if pre + SUF_A in W[n]]
        Bs = [W[n][pre + SUF_B].float() * (w * SCALE) for n, w in weights.items() if pre + SUF_A in W[n]]
        A, B = torch.cat(As, 0), torch.cat(Bs, 1); pad = R_MERGED - A.shape[0]
        if pad: A = torch.cat([A, A.new_zeros(pad, A.shape[1])], 0); B = torch.cat([B, B.new_zeros(B.shape[0], pad)], 1)
        out[pre + SUF_A], out[pre + SUF_B] = A.contiguous(), B.contiguous()
    os.makedirs(out_dir, exist_ok=True); save_file(out, f"{out_dir}/adapter_model.safetensors")
    LoraConfig(r=R_MERGED, lora_alpha=R_MERGED, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM",
               target_modules=sorted({p.split(".")[-1] for p in prefixes}), base_model_name_or_path=BASE).save_pretrained(out_dir)
    return out_dir

# ---------- compiler + cache ----------
CACHE = {}                                   # combo key -> loaded adapter name
def compile_fragments(ids):
    if not ids: return None, 0.0, "base"
    key = "+".join(sorted(ids))
    if key in CACHE: return CACHE[key], 0.0, "cache-hit"
    t0 = time.time()
    if len(ids) == 1:
        name = ids[0]                                                  # original adapter, already loaded
    else:
        name = "c_" + "_".join(sorted(ids))
        d = build_concat({i: REG[i]["merge_weight"] for i in ids}, str(WORK / "compiled" / name))
        model.load_adapter(d, adapter_name=name, is_trainable=False)
    CACHE[key] = name
    return name, time.time() - t0, "compiled"

# ---------- prompt formats (registry ke prompt_format ke hisaab se) ----------
def chat(u): return tok.apply_chat_template([{"role": "user", "content": u}], tokenize=False, add_generation_prompt=True)
def ni_wrap(example_input):                  # SuperNI: prefix + naya input + suffix
    i = example_input.rfind("Input: "); j = example_input.find("\nOutput", i)
    return example_input[:i + len("Input: ")], example_input[j:]
SQL_PRE, SQL_SUF = ni_wrap(sql_tr[0]["input"])
SAM_PRE, SAM_SUF = ni_wrap(load_dataset("Lots-of-LoRAs/task1572_samsum_summary")["train"][0]["input"])

FORMAT = {   # primary skill -> (prompt banane ka tareeka, add_special_tokens, max_new_tokens)
    "json":   (lambda q: chat(q), True, 400),
    "tone":   (lambda q: chat(q), True, 250),
    "sql":    (lambda q: chat(SQL_PRE + q + SQL_SUF), True, 200),
    "samsum": (lambda q: chat(SAM_PRE + ", ".join(l.strip() for l in q.splitlines() if l.strip()) + SAM_SUF), False, 100),
    None:     (lambda q: chat(q), True, 200),
}

@torch.no_grad()
def generate(prompt, add_special, max_new):
    x = tok(prompt, return_tensors="pt", add_special_tokens=add_special).to(model.device)
    y = model.generate(**x, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.eos_token_id, eos_token_id=tok.eos_token_id)
    return tok.decode(y[0][x["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def answer(query, force=None):
    """force=None -> router decide karega; force=[...] -> diye gaye adapters (oracle/baselines ke liye)."""
    t0 = time.time()
    ids, probs = (route(query) if force is None else (list(force), {}))
    t_route = time.time() - t0
    name, t_compile, how = compile_fragments(ids)
    fmt, sp, mx = FORMAT[ids[0] if ids else None]
    t1 = time.time()
    if name is None:
        with model.disable_adapter(): out = generate(fmt(query), sp, mx)
    else:
        model.set_adapter(name); out = generate(fmt(query), sp, mx)
    return {"adapters": ids, "compile": how, "t_route_ms": round(t_route * 1000, 1),
            "t_compile_s": round(t_compile, 2), "t_gen_s": round(time.time() - t1, 2), "output": out}

# ---------- demo: har tarah ki ek query ----------
DEMO = {
    "json":    TEST["json"][0],
    "sql":     TEST["sql"][0],
    "samsum":  TEST["samsum"][0],
    "tone":    TEST["tone"][0],
    "general": "What are three tips for staying focused while studying?",
    "mixed (tone+json)": "Subject: Wrong item delivered\n\nHi, I ordered a blue kettle (order #4521) but got a red toaster. "
                         "I'm really upset. Please reply kindly and also return the details as JSON with fields "
                         "order_id, ordered_item, received_item.",
}
for label, qtxt in DEMO.items():
    r = answer(qtxt)
    print(f"\n=== {label} ===  adapters={r['adapters'] or ['(base)']} | {r['compile']} | route {r['t_route_ms']} ms "
          f"| compile {r['t_compile_s']} s | gen {r['t_gen_s']} s")
    print(r["output"][:300])
r = answer(DEMO["mixed (tone+json)"], force=["json", "tone"])     # same combo dobara -> cache hit
print(f"\n=== mixed, forced json+tone (2nd time) === {r['compile']} | compile {r['t_compile_s']} s\n{r['output'][:400]}")

README.md:   0%|          | 0.00/2.77k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  114kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 27.7kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/valid-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 27.2kB            

data/valid-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating valid split: 0 examples [00:00, ? examples/s]


=== json ===  adapters=[np.str_('json')] | compiled | route 13.7 ms | compile 0.0 s | gen 32.35 s
{"orderID": 1001, "customer": {"partyID": 1, "name": "Acme Corporation", "email": "acme@example.com", "phone": "555-123-4567", "address": "123 Main Street", "city": "Anytown", "state": "CA", "zip": "91234"}, "product": {"productID": 2001, "name": "Widget", "description": "A small, plastic widget", "

=== sql ===  adapters=[np.str_('sql')] | compiled | route 8.4 ms | compile 0.0 s | gen 5.01 s
SELECT T1.character ,  T2.duration FROM actor AS T1 JOIN character AS T2 ON T1.actor_id  =  T2.actor_id

=== samsum ===  adapters=[np.str_('samsum')] | compiled | route 8.4 ms | compile 0.0 s | gen 2.36 s
Linda is cooking curry. Claire is eating a delicious looking dish.

=== tone ===  adapters=[np.str_('tone')] | compiled | route 9.3 ms | compile 0.0 s | gen 3.43 s
Your replacement speaker will be shipped out as soon as we receive the defective unit. You can keep the old one.

=== general ===  adapt

In [5]:
_route_orig = route
def route(query, thr=0.30, k=2):                       # np.str_ -> str (display fix)
    ids, p = _route_orig(query, thr, k); return [str(i) for i in ids], p

# ---------- scoring ----------
from collections import Counter
from rouge_score import rouge_scorer
R = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
def first_json(t):
    t = re.sub(r"```(json)?", "", t); i = t.find("{")
    if i < 0: return None
    try: return json.JSONDecoder().raw_decode(t[i:])[0]
    except Exception: return None
def flat(o, p=""):
    if isinstance(o, dict): return [x for k, v in o.items() for x in flat(v, f"{p}.{k}")]
    if isinstance(o, list): return [x for v in o for x in flat(v, p + "[]")]
    return [] if o in (None, "", []) else [(p, str(o).strip().lower())]
def json_f1(ref, pred):
    ref = json.loads(ref) if isinstance(ref, str) else ref
    pr = first_json(pred); a = Counter(flat(pr)) if pr is not None else Counter(); b = Counter(flat(ref))
    tp = sum((a & b).values())
    if tp == 0: return 0.0
    p, r = tp / sum(a.values()), tp / sum(b.values()); return 2 * p * r / (p + r)
SCORE = {"json": json_f1, "sql": lambda r, p: R.score(r, p)["rougeL"].fmeasure,
         "samsum": lambda r, p: R.score(r, p)["rougeL"].fmeasure}

# ---------- query sets (query, reference) ----------
N = {"json": 20, "sql": 30, "samsum": 30, "tone": 50, "general": 30}
jid = [i for i in range(len(EVAL["json"])) if i != 56][:N["json"]]
QS = {"json":   [(q_json(EVAL["json"][i]), EVAL["json"][i]["item"]) for i in jid],
      "sql":    [(q_sql(x), x["output"][0] if isinstance(x["output"], list) else x["output"]) for x in EVAL["sql"].select(range(N["sql"]))],
      "samsum": [(q_samsum(x["dialogue"]), x["summary"]) for x in EVAL["samsum"].select(range(N["samsum"]))],
      "tone":   [(q_tone(x), x["answer"]) for x in EVAL["tone"].select(range(N["tone"]))],
      "general":[(q_none(x), None) for x in dolly.select(range(NTR + NTE, NTR + NTE + N["general"]))]}

# ---------- runners ----------
def run_gen(query, ids, fmt_skill):
    name, _, _ = compile_fragments(ids); fmt, sp, mx = FORMAT[fmt_skill]
    if name is None:
        with model.disable_adapter(): return generate(fmt(query), sp, mx)
    model.set_adapter(name); return generate(fmt(query), sp, mx)

@torch.no_grad()
def run_loss(query, ref, ids, fmt_skill):
    name, _, _ = compile_fragments(ids)
    p = tok(FORMAT[fmt_skill][0](query), add_special_tokens=False)["input_ids"]
    a = tok(ref.strip(), add_special_tokens=False)["input_ids"] + [tok.eos_token_id]
    ids_t = torch.tensor([p + a], device=model.device); lab = ids_t.clone(); lab[0, :len(p)] = -100
    if name is None:
        with model.disable_adapter(): return model(input_ids=ids_t, labels=lab).loss.item()
    model.set_adapter(name); return model(input_ids=ids_t, labels=lab).loss.item()

SETUPS = {"base": lambda s: [], "all4": lambda s: list(NAMES), "oracle": lambda s: [s]}   # system alag se
E2E = {s: {} for s in QS}; ROUTES = {s: [] for s in QS}
torch.cuda.reset_peak_memory_stats()

for skill, items in QS.items():
    for qi, (qtxt, ref) in enumerate(tqdm(items, desc=skill)):
        sys_ids, _ = route(qtxt); ROUTES[skill].append(sys_ids)
        if skill == "general": continue
        row = {}
        for setup, ids_fn in SETUPS.items():
            ids = ids_fn(skill)
            if skill == "tone": row[setup] = run_loss(qtxt, ref, ids, skill)
            else:               row[setup] = SCORE[skill](ref, run_gen(qtxt, ids, skill))
        fmt_sys = sys_ids[0] if sys_ids else None
        if sys_ids == [skill]:   row["system"] = row["oracle"]          # same input -> same output
        elif skill == "tone":    row["system"] = run_loss(qtxt, ref, sys_ids, fmt_sys)
        else:                    row["system"] = SCORE[skill](ref, run_gen(qtxt, sys_ids, fmt_sys))
        E2E[skill][qi] = row

# ---------- summary ----------
print(f"\n{'task':8} {'base':>7} {'all4':>7} {'system':>7} {'oracle':>7} | {'GR all4':>8} {'GR system':>10} | routed right")
summary = {}
for skill in ["json", "sql", "samsum", "tone"]:
    m = {s: st.mean(r[s] for r in E2E[skill].values()) for s in ["base", "all4", "system", "oracle"]}
    sign = -1 if skill == "tone" else 1                                   # tone: loss, kam = achha
    gr = lambda x: (sign * (m[x] - m["base"])) / (sign * (m["oracle"] - m["base"]))
    right = sum(ids == [skill] for ids in ROUTES[skill]) / len(ROUTES[skill])
    summary[skill] = {**m, "gr_all4": gr("all4"), "gr_system": gr("system"), "routed_right": right}
    print(f"{skill:8} {m['base']:7.3f} {m['all4']:7.3f} {m['system']:7.3f} {m['oracle']:7.3f} | "
          f"{gr('all4'):8.2f} {gr('system'):10.2f} | {right:.2f}")
gen_base = sum(ids == [] for ids in ROUTES["general"]) / len(ROUTES["general"])
print(f"\ngeneral queries -> base model: {gen_base:.2f}")
print(f"AVG GR  all4 {st.mean(v['gr_all4'] for v in summary.values()):.2f} | system {st.mean(v['gr_system'] for v in summary.values()):.2f}")

# ---------- efficiency ----------
mb = lambda p: sum(f.stat().st_size for f in Path(p).rglob("*.safetensors")) / 1e6
eff = {"peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
       "adapter_mb": {n: round(mb(AD / n), 1) for n in NAMES},
       "merged_all4_mb": round(mb(WORK / "compiled" / "c_json_samsum_sql_tone"), 1) if (WORK / "compiled" / "c_json_samsum_sql_tone").exists() else None,
       "cache": list(CACHE)}
print("\nefficiency:", eff)
json.dump({"summary": summary, "general_to_base": gen_base, "routes": ROUTES, "rows": E2E, "efficiency": eff},
          open(WORK / "results" / "e2e.json", "w"), indent=1, default=str)
shutil.copy2(WORK / "registry.json", WORK / "results" / "registry.json")
shutil.copy2(WORK / "router.pkl", WORK / "results" / "router.pkl")
shutil.make_archive(str(WORK / "phase3_results"), "zip", WORK / "results")
print("saved -> /kaggle/working/phase3_results.zip")

json:   0%|          | 0/20 [00:00<?, ?it/s]

sql:   0%|          | 0/30 [00:00<?, ?it/s]

samsum:   0%|          | 0/30 [00:00<?, ?it/s]

tone:   0%|          | 0/50 [00:00<?, ?it/s]

general:   0%|          | 0/30 [00:00<?, ?it/s]


task        base    all4  system  oracle |  GR all4  GR system | routed right
json       0.322   0.576   0.606   0.606 |     0.89       1.00 | 1.00
sql        0.491   0.742   0.658   0.692 |     1.25       0.83 | 0.93
samsum     0.325   0.400   0.427   0.432 |     0.70       0.95 | 0.97
tone       2.729   1.348   1.622   1.622 |     1.25       1.00 | 1.00

general queries -> base model: 0.77
AVG GR  all4 1.02 | system 0.95

efficiency: {'peak_gpu_gb': 7.74, 'adapter_mb': {'json': 167.8, 'tone': 167.8, 'sql': 37.8, 'samsum': 37.8}, 'merged_all4_mb': 671.1, 'cache': ['json', 'sql', 'samsum', 'tone', 'json+tone', 'json+samsum+sql+tone']}
saved -> /kaggle/working/phase3_results.zip
